# 逻辑回归：用概率做决策的线性分类器
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/分类 | 源文件：逻辑回归.py | 核心功能：二分类/多分类、正则化对比、L1 特征选择

## 概述

逻辑回归（Logistic Regression）是机器学习中最经典的分类算法之一，名字里有"回归"二字，但它做的是**分类**。核心思想：在线性回归的基础上套一个 Sigmoid 函数，把输出映射到 [0, 1] 区间，解释为概率。

尽管名字朴素，逻辑回归在工业界的地位不可撼动——它是广告点击率预测、信用评分、医学诊断等场景的**基线模型**。原因很简单：可解释性强（系数直接反映特征影响）、训练快、输出概率（不只是类别）。

这个脚本演示了二分类和多分类场景、不同正则化参数的影响，以及 L1 正则化如何实现特征选择。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# --- 导入库 ---
from sklearn.preprocessing import StandardScaler

## 数学原理

### 1. Sigmoid 函数与二分类模型

**代码对应**：`LogisticRegression(C=1.0, solver="lbfgs")` 训练逻辑回归模型。

逻辑回归通过 Sigmoid 函数将线性输出映射为概率：

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \quad z = \mathbf{w}^T\mathbf{x} + b$$

Sigmoid 的性质：$\sigma(z) \in (0, 1)$，$\sigma'(z) = \sigma(z)(1 - \sigma(z))$，$\sigma(-z) = 1 - \sigma(z)$。

二分类的概率模型：

$$P(y=1|\mathbf{x}) = \sigma(\mathbf{w}^T\mathbf{x} + b), \quad P(y=0|\mathbf{x}) = 1 - \sigma(\mathbf{w}^T\mathbf{x} + b)$$

### 2. 最大似然估计与交叉熵损失

**代码对应**：sklearn 内部使用交叉熵（对数似然）作为损失函数。

对数似然函数：

$$\ell(\mathbf{w}) = \sum_{i=1}^{n}\left[y_i \ln P(y_i=1|\mathbf{x}_i) + (1-y_i)\ln P(y_i=0|\mathbf{x}_i)\right]$$

$$= \sum_{i=1}^{n}\left[y_i \ln \sigma(\mathbf{w}^T\mathbf{x}_i) + (1-y_i)\ln(1 - \sigma(\mathbf{w}^T\mathbf{x}_i))\right]$$

这等价于最小化**交叉熵损失**：

$$J(\mathbf{w}) = -\frac{1}{n}\ell(\mathbf{w}) = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \ln \hat{p}_i + (1-y_i)\ln(1-\hat{p}_i)\right]$$

梯度：

$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{n}\mathbf{X}^T(\hat{\mathbf{p}} - \mathbf{y})$$

没有闭式解，需要迭代优化（L-BFGS、SGD 等）。

### 3. 多分类扩展：Softmax 回归

**代码对应**：代码中 Iris 3 类分类使用 `LogisticRegression` 的默认 OvR 或 multinomial 策略。

Softmax 回归（多项逻辑回归）将二分类推广到 $K$ 类：

$$P(y=k|\mathbf{x}) = \frac{\exp(\mathbf{w}_k^T\mathbf{x} + b_k)}{\sum_{j=1}^{K}\exp(\mathbf{w}_j^T\mathbf{x} + b_j)}$$

**代码对应**：`lr_multi.coef_.shape` 为 $(K, p)$，即每个类别一组系数。

多分类的交叉熵损失：

$$J = -\frac{1}{n}\sum_{i=1}^{n}\sum_{k=1}^{K}\mathbb{I}(y_i = k)\ln P(y_i=k|\mathbf{x}_i)$$

### 4. 正则化

**代码对应**：`C` 参数控制正则化强度，`l1_ratio` 控制 L1/L2 比例。

sklearn 的逻辑回归目标函数：

$$J(\mathbf{w}) = -\frac{1}{C}\ell(\mathbf{w}) + R(\mathbf{w})$$

其中 $R(\mathbf{w})$ 为正则化项：
- `l1_ratio=0`（默认 L2）：$R = \frac{1}{2}\|\mathbf{w}\|_2^2$
- `l1_ratio=1`（L1）：$R = \|\mathbf{w}\|_1$
- `0 < l1_ratio < 1`（ElasticNet）：$R = l1\_ratio \cdot \|\mathbf{w}\|_1 + \frac{1-l1\_ratio}{2}\|\mathbf{w}\|_2^2$

注意 sklearn 中 $C$ 是正则化强度的**倒数**：$C$ 越小，正则化越强。

### 5. 决策边界的几何解释

二分类的决策边界为 $P(y=1|\mathbf{x}) = 0.5$，即：

$$\mathbf{w}^T\mathbf{x} + b = 0$$

这是一个 $(p-1)$ 维超平面。样本到决策边界的距离为：

$$d = \frac{|\mathbf{w}^T\mathbf{x} + b|}{\|\mathbf{w}\|_2}$$

Sigmoid 输出的概率可以理解为到决策边界的"置信度"——距离越远，概率越接近 0 或 1。

## 代码结构

| 段落 | 内容 |
|------|------|
| 数据准备 | 二分类合成数据 + Iris 多分类数据，均做 StandardScaler |
| 二分类 | LogisticRegression 训练、预测、概率输出、系数解读 |
| 多分类 | Iris 3 分类，OvR（One-vs-Rest）策略 |
| 正则化对比 | C 值从 0.01 到 100 的准确率变化 |
| L1 正则化 | 稀疏系数，自动特征选择 |

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
# 二分类场景
X_bin, y_bin = make_classification(
    n_samples=300, n_features=10, n_informative=5,
    n_classes=2, random_state=42
)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
# --- 赋值/配置 ---
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# 多分类场景（Iris）
iris = load_iris()
X_multi, y_multi = iris.data, iris.target
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

# 特征缩放（逻辑回归对特征尺度敏感）
scaler_b = StandardScaler()
X_train_b = scaler_b.fit_transform(X_train_b)
X_test_b = scaler_b.transform(X_test_b)

scaler_m = StandardScaler()
X_train_m = scaler_m.fit_transform(X_train_m)
X_test_m = scaler_m.transform(X_test_m)

### 2. 二分类

在分类任务上训练模型并评估性能。

In [ ]:
# solver: 'lbfgs'(小数据), 'liblinear'(小数据+L1), 'saga'(大数据+弹性网)
# C 越小正则化越强；l1_ratio=0 为 L2, l1_ratio=1 为 L1
lr_bin = LogisticRegression(C=1.0, l1_ratio=0, solver="lbfgs", max_iter=1000, random_state=42)
lr_bin.fit(X_train_b, y_train_b)

y_pred_b = lr_bin.predict(X_test_b)
y_prob_b = lr_bin.predict_proba(X_test_b)

print("=== 二分类逻辑回归 ===")
print(f"训练集准确率: {lr_bin.score(X_train_b, y_train_b):.4f}")
print(f"测试集准确率: {accuracy_score(y_test_b, y_pred_b):.4f}")
print(f"模型系数: {lr_bin.coef_.round(4)}")
print(f"截距: {lr_bin.intercept_.round(4)}")
# --- 输出结果 ---
print(f"\n混淆矩阵:\n{confusion_matrix(y_test_b, y_pred_b)}")
print(f"\n分类报告:\n{classification_report(y_test_b, y_pred_b)}")

### 3. 多分类（Iris）

在分类任务上训练模型并评估性能。

In [ ]:
lr_multi = LogisticRegression(C=1.0, l1_ratio=0, solver="lbfgs", max_iter=200, random_state=42)
lr_multi.fit(X_train_m, y_train_m)

y_pred_m = lr_multi.predict(X_test_m)
print("\n=== 多分类逻辑回归 (Iris) ===")
print(f"训练集准确率: {lr_multi.score(X_train_m, y_train_m):.4f}")
print(f"测试集准确率: {accuracy_score(y_test_m, y_pred_m):.4f}")
print(f"类别: {iris.target_names}")
# --- 输出结果 ---
print(f"模型系数形状: {lr_multi.coef_.shape} (类别数 × 特征数)")
print(f"\n分类报告:\n{classification_report(y_test_m, y_pred_m, target_names=iris.target_names)}")

### 4. 不同正则化对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 正则化强度对比 (C 值) ===")
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    lr_c = LogisticRegression(C=C, l1_ratio=0, solver="lbfgs", max_iter=200, random_state=42)
    lr_c.fit(X_train_m, y_train_m)
    train_acc = lr_c.score(X_train_m, y_train_m)
# --- 继续 ---
    test_acc = lr_c.score(X_test_m, y_test_m)
    print(f"C={C:>6}: 训练准确率={train_acc:.4f}, 测试准确率={test_acc:.4f}")

### 5. L1 正则化（特征选择）

运行 5. L1 正则化（特征选择） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== L1 正则化（稀疏系数）===")
lr_l1 = LogisticRegression(C=1.0, l1_ratio=1, solver="liblinear", max_iter=200, random_state=42)
lr_l1.fit(X_train_b, y_train_b)
n_nonzero = np.count_nonzero(lr_l1.coef_)
print(f"非零系数个数: {n_nonzero} / {lr_l1.coef_.shape[1]}")
# --- 输出结果 ---
print(f"系数: {lr_l1.coef_.round(4)}")
print(f"测试准确率: {lr_l1.score(X_test_b, y_test_b):.4f}")

## 关键代码解释

### 特征缩放是必须的

```python
scaler_b = StandardScaler()
X_train_b = scaler_b.fit_transform(X_train_b)
X_test_b = scaler_b.transform(X_test_b)
```

逻辑回归使用梯度下降求解，特征尺度差异大会导致收敛缓慢。**必须**在训练集上 fit，测试集上 transform。

### solver 参数的选择

```python
lr_bin = LogisticRegression(C=1.0, l1_ratio=0, solver="lbfgs", max_iter=1000, random_state=42)
```

solver 决定优化算法：
- lbfgs：准牛顿法，适合中小数据集，支持 L2 正则化
- liblinear：坐标下降法，适合小数据集，支持 L1 正则化
- saga：随机平均梯度下降，适合大数据集 + 弹性网

### C 参数与正则化强度

```python
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    lr_c = LogisticRegression(C=C, ...)
```

C 是正则化强度的**倒数**——C 越小，正则化越强，模型越简单。C 很大时接近无正则化，容易过拟合。

### L1 正则化做特征选择

```python
lr_l1 = LogisticRegression(C=1.0, l1_ratio=1, solver="liblinear", ...)
```

L1 正则化（Lasso）会把不重要特征的系数压缩到**正好为 0**，从而实现自动特征选择。非零系数个数直接反映模型认为重要的特征数量。

## 使用示例

```python
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000))
])
pipe.fit(X_train, y_train)
print(pipe.predict_proba(X_test))  # 输出概率
```

## 注意事项

1. **必须特征缩放**：逻辑回归基于梯度优化，特征尺度敏感
2. **C 的调优范围**：通常在 [0.001, 1000] 的对数网格上搜索
3. **多分类策略**：默认 OvR，multi_class="multinomial" 使用 Softmax
4. **不擅长非线性**：逻辑回归是线性决策边界，非线性问题需要特征工程或换模型
5. **概率校准**：输出的概率不一定准确，可以通过 Platt Scaling 或 Isotonic Regression 校准

## 总结与延伸

以上代码展示了 **逻辑回归** 的完整流程。

**解读要点**：
- 关注 **准确率（Accuracy）** 和 **分类报告** 中的精确率/召回率/F1
- 如果训练准确率远高于测试准确率，说明存在过拟合
- 观察 **混淆矩阵**，找出模型最容易混淆的类别

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Softmax 回归**：多分类的逻辑回归推广，直接优化交叉熵损失
- **正则化的贝叶斯解释**：L2 正则化等价于高斯先验，L1 正则化等价于拉普拉斯先验
- **逻辑回归 + 特征交叉**：加入特征交互项可以捕获非线性关系，这就是 FFM/DeepFM 等推荐模型的基础
- **在线学习**：SGDClassifier(loss="log_loss") 可以实现逻辑回归的在线版本
